# Getting Started with HMM Research Environment

This tutorial provides a comprehensive introduction to the HMM Research Environment. You'll learn how to:

1. Load and preprocess LDC signal data
2. Train Hidden Markov Models for regime detection
3. Analyze and visualize market regimes
4. Evaluate model performance
5. Save artifacts for production deployment

## Prerequisites

Make sure you have installed all required dependencies:

```bash
cd py
pip install -e ".[research]"
```

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# HMM Research Environment imports
from imp.data.ldc_loader import LDCDataLoader
from imp.data.validator import DataValidator
from imp.data.preprocessor import SignalPreprocessor
from imp.hmm.trainer import EnhancedHMMTrainer
from imp.hmm.inference import HMMInference
from imp.hmm.regime_analysis import RegimeAnalyzer
from imp.visualization.regime_visualizer import RegimeVisualizer
from imp.evaluation.evaluator import HMMEvaluator
from imp.hmm.artifact_management import ArtifactManager

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

## 2. Load and Validate Data

First, let's load LDC signals from the processed data.

In [ ]:
# Load LDC signals
loader = LDCDataLoader()

# Specify the signals we want to use
signals_to_load = ['s_LDC', 's_MR', 's_TSMOM']

# Load data
df = loader.load_signals(
    'processed_data/signals_processed.parquet',
    signals=signals_to_load
)

print(f"Loaded {len(df)} samples")
print(f"Signals: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Validate data quality
validator = DataValidator()
validation_report = validator.validate(df)

if validation_report['is_valid']:
    print("✓ Data validation passed")
else:
    print("⚠ Data quality issues found:")
    for issue in validation_report['issues']:
        print(f"  - {issue}")

# Display basic statistics
print("\nData Statistics:")
df.describe()

## 3. Preprocess Data

Preprocessing is crucial for HMM training. We'll normalize the data and handle any outliers.

In [ ]:
# Create preprocessor
preprocessor = SignalPreprocessor()

# Preprocess data
observations, preprocessing_metadata = preprocessor.preprocess(
    df,
    handle_missing='forward_fill',
    handle_outliers=True,
    normalize=True,
    outlier_threshold=3.0
)

print(f"Preprocessed data shape: {observations.shape}")
print(f"\nPreprocessing metadata:")
print(f"  Missing values handled: {preprocessing_metadata['missing_handled']}")
print(f"  Outliers removed: {preprocessing_metadata['outliers_removed']}")
print(f"  Normalization applied: {preprocessing_metadata['normalized']}")

# Visualize preprocessed data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, signal in enumerate(signals_to_load):
    axes[i].plot(observations[:500, i])
    axes[i].set_title(f'{signal} (Preprocessed)')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Normalized Value')
plt.tight_layout()
plt.show()

## 4. Split Data

Split data chronologically into training and test sets.

In [ ]:
# Split data (80% train, 20% test)
split_idx = int(len(observations) * 0.8)

train_data = observations[:split_idx]
test_data = observations[split_idx:]

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Features: {train_data.shape[1]}")

## 5. Train HMM Model

Now let's train a Hidden Markov Model to detect market regimes.

In [ ]:
# Create trainer
trainer = EnhancedHMMTrainer(
    n_states=3,  # Try to detect 3 market regimes
    library='hmmlearn',
    covariance_type='full',
    random_state=42
)

print("Training HMM model...")

# Train with validation
artifact, metrics = trainer.train_with_validation(
    train_data,
    validation_split=0.2,
    n_iterations=100
)

print("\n✓ Training complete!")
print(f"\nValidation Metrics:")
print(f"  Log-likelihood: {metrics['log_likelihood']:.4f}")
print(f"  AIC: {metrics['aic']:.4f}")
print(f"  BIC: {metrics['bic']:.4f}")

## 6. Perform Inference

Use the trained model to detect regimes in the test set.

In [ ]:
# Create inference engine
inference = HMMInference(artifact)

# Predict state probabilities
state_probs = inference.predict_proba(test_data)
print(f"State probabilities shape: {state_probs.shape}")

# Predict most likely state sequence
state_sequence = inference.predict(test_data)
print(f"State sequence shape: {state_sequence.shape}")

# Calculate test set score
test_score = inference.score(test_data)
print(f"\nTest set log-likelihood: {test_score:.4f}")

## 7. Visualize Regimes

Visualize the detected market regimes.

In [ ]:
# Create visualizer
visualizer = RegimeVisualizer(artifact)

# Plot state probabilities
fig = visualizer.plot_state_probabilities(
    state_probs[:500],  # Plot first 500 samples
    interactive=False,
    title='Market Regime Detection - Test Set'
)
plt.show()

In [ ]:
# Plot transition matrix
fig = visualizer.plot_transition_matrix(annotate=True)
plt.show()

print("\nTransition Matrix Interpretation:")
print("Diagonal values show regime persistence (higher = more stable)")
print("Off-diagonal values show transition probabilities between regimes")

## 8. Analyze Regimes

Perform detailed regime analysis to understand the characteristics of each market state.

In [ ]:
# Create regime analyzer
analyzer = RegimeAnalyzer(artifact)

# Perform comprehensive analysis
regime_analysis = analyzer.analyze_regimes(
    test_data,
    state_probs,
    feature_names=signals_to_load
)

print("Regime Analysis Results:\n")

# Display state durations
print("State Durations:")
for state, stats in regime_analysis['state_durations'].items():
    print(f"  State {state}:")
    print(f"    Mean duration: {stats['mean']:.1f} periods")
    print(f"    Median duration: {stats['median']:.1f} periods")
    print(f"    Max duration: {stats['max']:.0f} periods")

# Display transition frequencies
print("\nTransition Frequencies:")
for transition, count in regime_analysis['transition_frequencies'].items():
    print(f"  {transition}: {count} times")

In [ ]:
# Calculate state statistics
state_stats = analyzer.calculate_state_statistics(test_data, state_sequence)

print("State Characteristics:\n")
for state, stats in state_stats.items():
    print(f"State {state}:")
    print(f"  Mean: {stats['mean']}")
    print(f"  Std: {stats['std']}")
    print(f"  Volatility: {stats['volatility']:.4f}")
    print()

In [ ]:
# Get regime interpretation
interpretations = analyzer.get_regime_interpretation(state_stats)

print("Regime Interpretations:\n")
for state, interpretation in interpretations.items():
    print(f"State {state}: {interpretation}")

## 9. Model Evaluation

Evaluate model performance using cross-validation.

In [ ]:
# Create evaluator
evaluator = HMMEvaluator()

# Configuration for cross-validation
config = {
    'n_states': 3,
    'library': 'hmmlearn',
    'covariance_type': 'full'
}

print("Running cross-validation...")

# Perform cross-validation
cv_results = evaluator.cross_validate(
    train_data,
    trainer_config=config,
    cv_folds=5
)

print("\nCross-Validation Results:")
print(f"  Mean score: {cv_results['mean_score']:.4f}")
print(f"  Std score: {cv_results['std_score']:.4f}")
print(f"  Fold scores: {[f'{s:.4f}' for s in cv_results['fold_scores']]}")

## 10. Save Artifact

Save the trained model artifact for production deployment.

In [ ]:
from datetime import datetime

# Create artifact manager
manager = ArtifactManager(artifacts_dir='../artifacts')

# Prepare comprehensive metadata
metadata = {
    'training_date': datetime.now().isoformat(),
    'data_source': 'LDC signals',
    'signals_used': signals_to_load,
    'training_samples': len(train_data),
    'test_samples': len(test_data),
    'training_config': {
        'n_states': 3,
        'library': 'hmmlearn',
        'covariance_type': 'full',
        'random_state': 42
    },
    'validation_metrics': {
        'log_likelihood': float(metrics['log_likelihood']),
        'aic': float(metrics['aic']),
        'bic': float(metrics['bic'])
    },
    'test_performance': {
        'log_likelihood': float(test_score)
    },
    'cv_results': {
        'mean_score': float(cv_results['mean_score']),
        'std_score': float(cv_results['std_score'])
    },
    'preprocessing': preprocessing_metadata,
    'regime_interpretation': interpretations,
    'created_by': 'tutorial_notebook'
}

# Validate artifact
validation_report = manager.validate_artifact(artifact, validation_data=test_data)

if validation_report['is_valid']:
    # Save artifact
    artifact_path = manager.save_artifact(
        artifact,
        name='tutorial_regime_detector',
        version='1.0.0',
        metadata=metadata
    )
    print(f"✓ Artifact saved to: {artifact_path}")
else:
    print("⚠ Artifact validation failed:")
    for error in validation_report['errors']:
        print(f"  - {error}")

## 11. Summary

Congratulations! You've completed the getting started tutorial. Here's what you learned:

1. ✓ Loading and validating LDC signal data
2. ✓ Preprocessing data for HMM training
3. ✓ Training Hidden Markov Models
4. ✓ Performing inference and regime detection
5. ✓ Visualizing market regimes
6. ✓ Analyzing regime characteristics
7. ✓ Evaluating model performance
8. ✓ Saving artifacts for production

## Next Steps

Explore these notebooks to learn more:

- `01_data_exploration.ipynb` - Deep dive into data exploration
- `02_hmm_training_comparison.ipynb` - Compare different HMM implementations
- `03_regime_analysis.ipynb` - Advanced regime analysis techniques
- `04_parameter_optimization.ipynb` - Hyperparameter tuning
- `05_parameter_tuning_demo.ipynb` - Interactive parameter tuning

## Resources

- [API Documentation](../py/docs/HMM_RESEARCH_API.md)
- [Best Practices](../py/docs/BEST_PRACTICES.md)
- [Troubleshooting Guide](../py/docs/TROUBLESHOOTING.md)
- [Integration Examples](../py/docs/INTEGRATION_EXAMPLES.md)